In [50]:
import pickle
import networkx as nx
import torch 
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings

from typing import Tuple
from tqdm import tqdm

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils import network_generator, sample_scoring

# Learning with Expression Data, Disease KG & Healthy Aging KG

1. Generate sample_2_kg network (sample_ad_kg, sample_healthy_kg as comparison)
   + logFC
   + ecdf
   + z-score
   + average
   + all
2. Networkx to HeteroData
3. Node classification

### 1. Sample_KG Network Generation

In [ ]:
df_adni = pd.read_csv("./data/ADNI/adni_gene_cleaned.csv", index_col=0)
adni_design = pd.read_csv("./data/ADNI/design_with_real_target.tsv", sep='\t', index_col=0)
adni_design['Target'] = adni_design['Target'].map({'Control':0, 'AD':1, 'MCI':2})

3 classes ADNI data: Control, AD, MCI

In [24]:
adni_design.to_csv("./data/ADNI/design_3cls.csv")
adni_labels_3cls = adni_design['Target'].to_list()
df_adni_3cls = df_adni.T
df_adni_3cls.to_csv("./data/ADNI/adni_exp_3cls.csv")
df_adni_3cls

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,3.651,2.2865,3.039,2.3395,2.783,2.25300,2.081,7.043,3.30950,2.202,...,5.748,3.77150,4.0365,4.353000,5.1763,1.9045,5.31750,9.2190,7.3010,5.7490
037_S_4410,3.183,2.1230,3.543,2.2085,2.383,2.37550,1.733,6.773,3.27625,2.317,...,5.974,4.17300,4.4415,4.520667,5.0964,1.9265,5.38800,8.3785,6.7580,6.0935
006_S_4153,3.278,2.3545,3.528,2.1745,2.593,2.46825,1.841,6.910,3.20875,2.540,...,5.119,3.91275,4.6210,4.230667,5.1143,2.2315,5.62800,9.1085,7.3365,5.2615
116_S_1232,3.371,2.3725,3.835,2.1545,2.570,2.51925,2.249,7.209,3.24950,2.559,...,4.904,3.73800,4.4435,4.050667,5.1520,2.0545,5.46300,9.3210,7.1685,4.7340
099_S_4205,3.358,2.3865,3.392,2.1720,2.660,2.39725,1.893,6.920,3.15425,2.347,...,5.533,3.94600,4.5215,4.639667,5.2103,2.0405,5.64750,9.0300,7.2025,5.4575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
009_S_2381,3.302,2.5075,3.524,2.2525,2.876,2.76275,2.089,6.805,3.09150,2.650,...,4.523,3.50150,4.4685,3.962333,5.0892,1.9320,5.44275,9.2900,6.7035,4.7915
053_S_4557,3.403,2.3090,3.515,2.3225,3.106,2.85125,2.102,7.265,3.27575,2.603,...,5.087,3.58325,4.1555,4.125667,5.0986,1.9445,5.14000,9.8090,7.2810,4.7055
073_S_4300,3.530,2.4155,3.651,2.0760,2.707,2.38325,2.092,7.375,3.26950,2.557,...,4.938,3.63450,4.5165,4.070333,5.1731,2.0795,5.38775,9.5215,6.9825,5.0785
041_S_4014,3.532,2.4545,3.609,2.3495,3.081,2.69125,2.024,7.257,3.11425,2.497,...,4.037,3.68525,4.0480,3.976333,4.8241,2.0075,5.05725,9.5810,6.6865,3.8660


2 classes ADNI data: Control, AD(Disease)

In [25]:
adni_design_2cls = adni_design[adni_design['Target'] != 2]
# save 
adni_design_2cls.to_csv("./data/ADNI/design_2cls.csv")

adni_labels_2cls = adni_design['Target'].to_list()
df_adni_2cls = df_adni[adni_design_2cls.index.to_list()].T
df_adni_2cls.to_csv("./data/ADNI/adni_exp_2cls.csv")
df_adni_2cls

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,3.651,2.2865,3.039,2.3395,2.783,2.25300,2.081,7.043,3.30950,2.202,...,5.748,3.77150,4.0365,4.353000,5.1763,1.9045,5.31750,9.2190,7.3010,5.7490
037_S_4410,3.183,2.1230,3.543,2.2085,2.383,2.37550,1.733,6.773,3.27625,2.317,...,5.974,4.17300,4.4415,4.520667,5.0964,1.9265,5.38800,8.3785,6.7580,6.0935
006_S_4153,3.278,2.3545,3.528,2.1745,2.593,2.46825,1.841,6.910,3.20875,2.540,...,5.119,3.91275,4.6210,4.230667,5.1143,2.2315,5.62800,9.1085,7.3365,5.2615
116_S_1232,3.371,2.3725,3.835,2.1545,2.570,2.51925,2.249,7.209,3.24950,2.559,...,4.904,3.73800,4.4435,4.050667,5.1520,2.0545,5.46300,9.3210,7.1685,4.7340
128_S_0205,3.194,2.3560,3.146,2.1230,2.673,2.52150,1.766,7.079,3.10525,2.448,...,5.571,4.07650,4.4430,4.366667,5.2429,1.9105,5.56125,8.5345,6.9175,5.6455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
014_S_4668,3.330,2.2820,3.497,2.2965,3.155,2.67350,2.158,6.950,3.33150,2.923,...,4.309,3.60500,4.2075,3.903667,4.9955,2.1810,5.41125,9.3105,6.7660,5.0440
130_S_0289,3.368,2.2950,3.128,1.9650,2.622,2.45225,1.953,6.878,3.10100,2.481,...,5.649,3.86475,4.6765,4.663333,5.1113,2.0300,5.36250,9.1965,7.1770,5.5510
009_S_2381,3.302,2.5075,3.524,2.2525,2.876,2.76275,2.089,6.805,3.09150,2.650,...,4.523,3.50150,4.4685,3.962333,5.0892,1.9320,5.44275,9.2900,6.7035,4.7915
041_S_4014,3.532,2.4545,3.609,2.3495,3.081,2.69125,2.024,7.257,3.11425,2.497,...,4.037,3.68525,4.0480,3.976333,4.8241,2.0075,5.05725,9.5810,6.6865,3.8660


geo dataset

In [ ]:
df_geo = pd.read_csv("./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv", index_col=0).T
df_geo

,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDB,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981
GSM1423780,-0.066694,0.023919,-0.031845,0.015204,0.089450,-0.167457,0.023215,0.004379,0.041729,0.074247,...,0.066588,-0.020308,0.080426,-0.057932,0.007106,0.106203,0.063151,0.066667,0.060402,0.023945
GSM1423781,0.054097,-0.012131,-0.030740,0.034282,-0.006780,0.068906,-0.004402,-0.029354,0.003009,-0.035774,...,0.035217,-0.000390,-0.027666,0.033120,0.065091,-0.040029,-0.118265,0.096145,-0.007896,0.022231
GSM1423782,0.025282,0.001929,-0.085577,-0.023002,-0.314342,0.092161,-0.043387,0.069851,0.010428,0.116397,...,0.072900,-0.042220,0.085496,-0.020298,0.040479,-0.146334,-0.027519,0.165024,-0.008110,0.118489
GSM1423783,0.002485,0.034625,-0.066813,-0.018289,0.094836,0.073653,0.073933,-0.043867,0.045298,-0.087155,...,-0.064955,-0.034116,-0.109619,0.029910,0.079794,-0.000814,-0.250529,-0.014006,-0.238408,-0.042106
GSM1423784,-0.092606,0.019591,-0.030884,0.020118,-0.206281,0.050492,-0.003392,0.055446,0.111875,0.152810,...,-0.091836,-0.050365,0.105040,0.052845,-0.013048,0.023013,0.051360,0.054292,0.115933,0.035485
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1424242,-0.035529,-0.067985,-0.115346,-0.045191,-0.085769,-0.041509,-0.018460,0.108033,0.010502,0.153964,...,-0.017694,-0.130394,0.012196,-0.022353,0.032304,-0.100798,0.016300,0.053928,-0.060836,0.176064
GSM1424243,0.075622,-0.010792,-0.088835,-0.021300,-0.096128,-0.037948,0.038506,0.107594,0.001454,0.062382,...,0.025705,-0.133761,-0.092569,0.121448,0.159210,0.017417,-0.108027,0.072586,0.090821,0.010657
GSM1424244,0.065244,-0.087919,-0.120331,-0.038014,0.068436,-0.003720,0.013911,0.034820,0.105181,0.085700,...,-0.043359,-0.048832,-0.040329,0.001067,0.012171,0.130757,0.016830,0.094532,-0.294262,0.086503
GSM1424245,0.054953,0.072740,-0.094495,-0.028996,-0.039163,-0.066074,0.028088,0.070063,-0.028701,0.052751,...,-0.040056,-0.091335,-0.080240,0.045302,0.084873,0.083839,-0.073553,0.106768,-0.118929,0.061755


In [ ]:
geo_design = pd.read_csv("./data/GEO/GSE33000_ad_hd/GSE33000_meta_2cls.csv", index_col=0)
geo_design['Target'] = geo_design['Target'].map({'Disease':1, 'Control':0})
geo_labels = geo_design['Target'].to_list()
geo_design

,Tissue,Sample type,Age,Gender,Disease status,Target
GSM1423780,prefrontal cortex brain,reference,67 yrs,female,Alzheimer's disease,1
GSM1423781,prefrontal cortex brain,reference,88 yrs,male,Alzheimer's disease,1
GSM1423782,prefrontal cortex brain,reference,62 yrs,male,Alzheimer's disease,1
GSM1423783,prefrontal cortex brain,reference,90 yrs,female,Alzheimer's disease,1
GSM1423784,prefrontal cortex brain,reference,90 yrs,female,Alzheimer's disease,1
...,...,...,...,...,...,...
GSM1424242,prefrontal cortex brain,reference,52 yrs,male,non-demented,0
GSM1424243,prefrontal cortex brain,reference,61 yrs,male,non-demented,0
GSM1424244,prefrontal cortex brain,reference,60 yrs,female,non-demented,0
GSM1424245,prefrontal cortex brain,reference,55 yrs,female,non-demented,0


In [43]:
def process_and_save(
    data: pd.DataFrame, 
    design: pd.DataFrame, 
    threshold: float, 
    control: str|int,
    do_function,
    output_dir: str,
    method: str = 'logfc'
):
    """
    Wrapper to run the search, show progress, and save results.
    """
    # Create directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    print(f"Starting {method} analysis...")
    
    # Simple progress bar for the high-level step
    with tqdm(total=2, desc="Overall Progress") as pbar:
        # 1. do sample scoring
        output_df, summary_df = do_function(data, design, threshold, control=control)
        
        pbar.update(1)
        
        # 2. Save the files
        out_path = os.path.join(output_dir, f"sample_scoring_{method}.csv")
        sum_path = os.path.join(output_dir, f"scoring_summary_{method}.csv")
        
        output_df.to_csv(out_path)
        summary_df.to_csv(sum_path)
        pbar.update(1)

    print(f"Done! Files saved to {output_dir}")
    return output_df, summary_df

#### (1) logFC
to decide the edges between sample and protein node

In [ ]:
def do_biological_logfc_search(
    data: pd.DataFrame,
    design: pd.DataFrame,
    threshold: float = 0.1,
    alpha: float = 0.05,
    control: str|int = 0
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Identifies 'Radicals' based on the Volcano Plot regions:
    1  (Red):  logFC > threshold  AND  adj.P-value < alpha
    -1 (Blue): logFC < -threshold AND  adj.P-value < alpha
    0  (Gray): Does not meet both criteria.
    """
    
    # 1. Safety Check & Log Transformation
    max_val = np.percentile(data.values, 99)
    working_data = np.log2(data + 1) if max_val > 50 else data.copy()
    if not isinstance(working_data, pd.DataFrame):
        working_data = pd.DataFrame(working_data, index=data.index, columns=data.columns)

    # 2. Group Identification
    # Transpose so samples are rows for easier grouping
    if len(data) != len(design):
        working_data = working_data.transpose() 

    control_idx = design[design['Target'] == control].index
    case_idx = design[design['Target'] != control].index
    
    # 3. Statistical Testing (Gene by Gene)
    results = []
    for gene in working_data.index:
        control_vals = working_data.loc[gene, control_idx]
        case_vals = working_data.loc[gene, case_idx]
        
        # Calculate Mean LogFC (Case Mean - Control Mean)
        mean_logfc = case_vals.mean() - control_vals.mean()
        
        # Perform T-Test (Comparing the two distributions)
        # equal_var=False performs Welch's T-test (safer for bio data)
        t_stat, p_val = stats.ttest_ind(case_vals, control_vals, equal_var=False)
        
        results.append({'gene': gene, 'logFC': mean_logfc, 'p_value': p_val})
    
    stats_df = pd.DataFrame(results).set_index('gene')
    
    # 4. Multiple Testing Correction (FDR / Benjamini-Hochberg)
    # This prevents false positives when testing thousands of genes
    stats_df['adj_P_val'] = multipletests(stats_df['p_value'], method='fdr_bh')[1]

    # 5. Scoring the Samples (The 'Volcano' Logic)
    # Initialize output matrix [samples x genes]
    output_df = pd.DataFrame(0, index=data_t.index, columns=working_data.index)
    
    # We only mark a gene as 1 or -1 if the GENE ITSELF is significant overall
    # and the individual sample's expression is extreme.
    for gene in working_data.index:
        gene_stats = stats_df.loc[gene]
        
        if gene_stats['adj_P_val'] < alpha:
            # Calculate sample-specific deviation from control mean
            ctrl_mean = working_data.loc[gene, control_idx].mean()
            sample_deviations = data_t[gene] - ctrl_mean
            
            # Upper-Red Region (Significant Up)
            output_df.loc[sample_deviations > threshold, gene] = 1
            # Upper-Blue Region (Significant Down)
            output_df.loc[sample_deviations < -threshold, gene] = -1

    # 6. Metadata and Summary
    label_mapping = {key: val for val, key in enumerate(np.unique(design['Target']))}
    output_df['label'] = design.loc[output_df.index, 'Target'].map(label_mapping)
    
    summary_df = stats_df.copy() # Summary now includes the actual stats
    
    return output_df, summary_df

In [52]:
def do_logfc(
    data: pd.DataFrame,
    design: pd.DataFrame,
    threshold: float,
    control: str|int
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Identify samples with extreme logFC values relative to the control mean.
    
    :param data: Dataframe [features x samples] gene expression values (assumed log-scaled)
    :param design: Dataframe [samples x info] containing the 'Target' column
    :param threshold: The logFC cutoff (e.g., 2.0 for a 4-fold change)
    :param control: The string or int label for the control group
    """

    # 1. Safety Check: Is the data log-scaled?
    max_val = np.percentile(data.values, 99)
    
    if max_val > 50:
        warnings.warn(f"Data appears to be raw counts (Max: {max_val:.2f}). Applying log2(x + 1) transformation.")
        # Apply log2 transformation: adding 1 avoids log(0) errors
        working_data = np.log2(data + 1)
        if not isinstance(working_data, pd.DataFrame):
            working_data = pd.DataFrame(working_data, index=data.index, columns=data.columns)
    else:
        working_data = data.copy()

    # 2. Align data and design
    if len(data) == len(design):
        data_t = working_data
    else:
        data_t = working_data.transpose() 

    # 3. Calculate the Mean of the Control Group for every gene
    # We use .loc to ensure we only average the samples labeled as 'control'
    control_samples = design[design['Target'] == control].index
    
    # Check if control samples exist in the data columns
    missing_controls = [c for c in control_samples if c not in data_t.index]
    if missing_controls:
        raise ValueError(f"Control samples {missing_controls} not found in data columns.")

    control_mean = data_t.loc[control_samples].mean()

    # 4. Calculate LogFC
    # Since data is log-scaled: LogFC = Patient - Control_Average
    logfc_matrix = data_t.sub(control_mean, axis=1)

    # 5. Score based on the threshold
    output_df = pd.DataFrame(0, index=logfc_matrix.index, columns=logfc_matrix.columns)
    output_df[logfc_matrix > threshold] = 1
    output_df[logfc_matrix < -threshold] = -1

    # 6. Map Labels
    label_mapping = {key: val for val, key in enumerate(np.unique(design['Target']))}
    # Ensure design is aligned with the output_df index
    output_df['label'] = design.loc[output_df.index, 'Target'].map(label_mapping)

    # 7. Summary
    summary_df = output_df.drop(columns=['label']).apply(pd.Series.value_counts).fillna(0)

    return output_df, summary_df

In [53]:
df_adni_logfc,summary_adni_logfc = process_and_save(
    data=df_adni_2cls, 
    design=adni_design_2cls, 
    threshold=0.1, 
    control=0,
    do_function=do_biological_logfc_search,
    output_dir='./data/ADNI',
    method = 'logfc'
)

Starting logfc analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]


KeyError: "None of [Index(['037_S_4410', '116_S_1232', '128_S_0205', '036_S_4491', '031_S_2018',\n       '067_S_4072', '037_S_4308', '018_S_4313', '067_S_0257', '073_S_4382',\n       ...\n       '129_S_4369', '137_S_0686', '011_S_4120', '041_S_0125', '052_S_0951',\n       '014_S_4401', '023_S_4164', '082_S_4339', '022_S_2379', '041_S_4014'],\n      dtype='str', name='FileName', length=222)] are in the [columns]"

In [45]:
summary_adni_logfc

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
-1,160,83,181,67,132,111,90,142,106,136,...,161,164,157,141,124,73,153,165,157,181
0,148,292,122,303,195,227,274,142,238,191,...,77,146,114,169,232,314,161,116,138,68
1,147,80,152,85,128,117,91,171,111,128,...,217,145,184,145,99,68,141,174,160,206


In [37]:
df_adni_logfc

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,label
116_S_1249,1,0,-1,1,0,-1,0,0,0,-1,...,0,-1,1,0,0,-1,-1,1,1,1
037_S_4410,-1,-1,1,0,-1,-1,-1,-1,0,-1,...,1,0,1,0,0,0,-1,-1,1,0
006_S_4153,0,0,1,0,-1,0,-1,0,0,0,...,1,1,0,0,1,1,-1,1,1,1
116_S_1232,0,0,1,0,-1,0,1,1,0,0,...,0,0,-1,0,0,0,0,1,-1,0
128_S_0205,-1,0,-1,-1,0,0,-1,0,-1,0,...,1,0,1,1,0,1,-1,-1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
014_S_4668,0,0,0,0,1,1,1,0,1,1,...,0,-1,-1,-1,1,0,0,-1,0,1
130_S_0289,0,0,-1,-1,0,0,0,-1,-1,0,...,1,1,1,0,0,0,-1,1,1,1
009_S_2381,0,1,1,0,1,1,0,-1,-1,1,...,-1,0,-1,0,0,0,0,-1,-1,1
041_S_4014,1,0,1,1,1,1,0,1,-1,0,...,0,-1,-1,-1,0,-1,1,-1,-1,0


In [49]:
df_geo_logfc,summary_geo_logfc = process_and_save(
    data=df_geo, 
    design=geo_design, 
    threshold=0.03, 
    control=1,
    do_function=do_logfc,
    output_dir='./data/GEO/GSE33000_ad_hd',
    method = 'logfc'
)
summary_geo_logfc

Starting logfc analysis...


Overall Progress: 100%|██████████| 2/2 [00:12<00:00,  6.22s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd


,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDB,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981
-1,139,200,259,115,282,170,208,112,176,121,...,106,168,140,161,191,185,235,137,156,121
0,175,146,96,255,36,160,149,68,182,54,...,198,138,52,154,134,97,105,69,119,69
1,153,121,112,97,149,137,110,287,109,292,...,163,161,275,152,142,185,127,261,192,277


#### (2) ECDF

#### (3) Average

### 2. Networkx to HeteroData